# Proposed validation — review before it runs

**What this measures:** The target `l0_deficit_from_k` measures, on real gpt2 `blocks.6.hook_resid_post` activations pushed through the actually-imported `AbsTopK` operator at the PR's pre-registered geometry (d_sae=8192, k=64, 500 token positions), that the new `!=0` L0 counting reports exactly k active features per token — a ReLU-style sign-kill or a k-violation reads as deficit ≈ k/2, so one number certifies both exactly-k magnitude selection and sign preservation (kind: capability — `main` cannot import `AbsTopK` at all, so `baseline: none` and no stand-in number is ever emitted). The guardrail `relu_topk_l0_predicate_delta` checks the PR's load-bearing safety claim that the `>0`→`!=0` predicate swap is an exact no-op (delta == 0.0) for ReLU-family (TopK) activations; the deferred W&B training-parity claim (MSE/CE vs TopK/JumpReLU) is explicitly out of scope and remains unmeasured here.

**Target metric:** `l0_deficit_from_k`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `benchmark/bench_abstopk_mechanism.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the script at `benchmark/bench_abstopk_mechanism.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "benchmark/bench_abstopk_mechanism.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
# ruff: noqa: T201
"""AbsTopK mechanism benchmark (capability question; baseline: none — the
comparison lives inside this one run: AbsTopK vs a ReLU-family control on
IDENTICAL pre-activations from real gpt2 residual-stream activations).

Exercises the real changed operator, ``sae_lens.saes.abstopk_sae.AbsTopK``
(full definition visible in the diff), at the PR's pre-registered geometry
(held_constant: gpt2, blocks.6.hook_resid_post, d_in=768, d_sae=8192, k=64,
first 500 token positions of a fixed text, seed-0 random encoder projection),
and measures the changed evals.py counting predicates (``> 0`` -> ``!= 0``) on
the real operator output (the run_evals/ActivationsStore loop itself is not
driven — its constructor signatures are outside the visible context). The
encoder is random-init, so NO reconstruction-quality numbers are printed.

Baseline arm: ``AbsTopK`` is a symbol the diff ADDS, so its import is guarded
with ImportError/AttributeError only. When the changed code is absent the
AbsTopK pathway degrades to the pre-change ReLU-family control activations
(abstopk_registered=0.0, old_predicate_undercount_ratio=1.0,
bidirectionality_deviation=0.5) instead of crashing; the measurement itself
(model load, encoding, scoring) is never wrapped and raises with a traceback
on failure, and the JSON line still prints unconditionally on both arms.

Prints exactly one JSON metrics line as the last line of stdout; deterministic
on CPU (counting metrics, no wall-clock timing).
"""
import argparse
import json
import os
import sys
from pathlib import Path

In [ ]:
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))  # repo root

import torch
from transformer_lens import HookedTransformer

# Symbol added by the diff: on the baseline arm this import must fail cleanly.
try:
    from sae_lens.saes.abstopk_sae import AbsTopK
except (ImportError, AttributeError):
    AbsTopK = None

In [ ]:
HOOK = "blocks.6.hook_resid_post"
K = 64
D_SAE = 8192
TEXT = (
    "The city council met on Thursday evening to review the railway proposal. "
    "Engineers presented maps, budgets, and a timeline for the northern branch. "
    "Residents asked whether the old station would keep its glass roof intact. "
    "A short debate followed about bicycles, buses, and the price of a ticket. "
    "The chair summarised the objections and deferred the vote until October. "
    "Outside, rain moved across the platform where the last train was waiting. "
    "Somewhere in the archive a clerk filed the minutes beside older papers. "
    "History, someone said, is mostly a record of postponed decisions. "
) * 4  # fixed text; comfortably > 500 gpt2 tokens

In [ ]:
def registry_check() -> float:
    """1.0 iff sae_lens.registry resolves architecture 'abstopk' to the new classes."""
    try:
        import sae_lens
        from sae_lens.registry import get_sae_class, get_sae_training_class

        if (
            get_sae_class("abstopk") is sae_lens.AbsTopKSAE
            and get_sae_training_class("abstopk") is sae_lens.AbsTopKTrainingSAE
        ):
            return 1.0
    except (ImportError, AttributeError, KeyError, ValueError):
        return 0.0
    return 0.0

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)  # accepted and ignored
    parser.add_argument("--ref", default=None)  # accepted and ignored
    parser.add_argument("--seed", type=int, default=0)  # accepted and ignored
    parser.parse_args()

    smoke = os.environ.get("REMYX_SMOKE", "0") == "1"
    n_tokens = 16 if smoke else 500
    torch.set_grad_enabled(False)

    model = HookedTransformer.from_pretrained("gpt2", device="cpu")
    _, cache = model.run_with_cache(TEXT, names_filter=[HOOK])
    acts = cache[HOOK][0, :n_tokens, :].float().cpu()  # (n_tokens, d_in=768)

    gen = torch.Generator(device="cpu").manual_seed(0)
    d_in = acts.shape[-1]
    w_enc = torch.randn(D_SAE, d_in, generator=gen) / (d_in**0.5)
    hidden_pre = acts @ w_enc.T  # (n_tokens, D_SAE); ~symmetric pre-activations

    # Pre-change (ReLU-family) control on the identical pre-activations:
    # top-k over non-negative activations, values >= 0 by construction.
    vals, idx = torch.topk(torch.relu(hidden_pre), K, dim=-1)
    relu_acts = torch.zeros_like(hidden_pre).scatter_(-1, idx, vals)

    if AbsTopK is not None:
        sae_acts = AbsTopK(K)(hidden_pre)  # the real changed operator
    else:
        # Baseline arm (changed code absent): degrade to the pre-change
        # ReLU-family behaviour — the identical control activations.
        sae_acts = relu_acts

    # The two changed evals.py predicates, applied exactly as written there.
    active = sae_acts != 0
    l0_new = active.sum(dim=-1).float().mean()
    l0_old = (sae_acts > 0).sum(dim=-1).float().mean()
    # For t >= 0, (t > 0) == (t != 0) elementwise, so correct arithmetic gives
    # EXACTLY 0.0 here. Guardrail bound 0.5 (half an active feature per token)
    # derivation: per-token counts are integers, so correct behaviour sits at
    # 0.0 while any wholesale predicate regression — wrong operator (>= 0
    # counts the 8128 padding zeros) or wrong tensor (< 0 counts nothing,
    # delta = the full k=64) — shifts mean L0 by >= 1.0 and fails the bound.
    ctrl_delta = abs(
        float((relu_acts != 0).sum(dim=-1).float().mean())
        - float((relu_acts > 0).sum(dim=-1).float().mean())
    )

    # Independent (argsort-based) reference for the k-largest-by-magnitude set.
    k_largest = torch.argsort(hidden_pre.abs(), dim=-1, descending=True)[:, :K]
    ref_mask = torch.zeros(active.shape, dtype=torch.bool)
    ref_mask.scatter_(1, k_largest, torch.ones_like(k_largest, dtype=torch.bool))

    neg_share = float((sae_acts < 0).sum()) / float(active.sum())

    # Undercount ratio = positive actives / all actives = 1 - neg_share, so the
    # expected value is ~0.5. Bound 0.9 derivation: it fails only when fewer
    # than 10% of active features are negative (neg_share < 0.10), i.e. when
    # the >0 -> !=0 fix would have corrected essentially nothing (ratio 1.0 =
    # sign-killing); a healthy bidirectional AbsTopK reads ~0.5 and passes.
    undercount_ratio = float(l0_old / l0_new)

    metrics = {
        # 0.0 expected: exactly k signed values scattered; deficit > 0 only if a
        # selected pre-activation is exactly 0.0 (measure-zero in fp32).
        "l0_deficit_from_k": float(K - l0_new),
        "relu_topk_l0_predicate_delta": ctrl_delta,
        # 3-sigma band 0.0084 at 500 tokens x k=64 = 32000 signs (sigma=0.002795).
        "bidirectionality_deviation": abs(neg_share - 0.5),
        # ~0.5 = the 2x undercount signature the eval fix corrects.
        "old_predicate_undercount_ratio": undercount_ratio,
        "magnitude_selection_exact": float(
            (active == ref_mask).all(dim=-1).float().mean()
        ),
        "sign_preservation_exact": float(
            (torch.sign(sae_acts[active]) == torch.sign(hidden_pre[active]))
            .float()
            .mean()
        ),
        "abstopk_registered": registry_check(),
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "@remyx-ai validate — does the AbsTopK port select exactly the k largest-magnitude pre-activations per token with signs preserved (~half negative) on real gpt2 activations, and is the >0 -> !=0 eval-counting fix an exact no-op for ReLU variants?"
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: abstopk-mechanism
    suite: "benchmark/bench_abstopk_mechanism.py"
    baseline: none
    scorer: l0_deficit_from_k
    policy: {guardrail_veto: true}
    compute: {tier: cpu}
    metrics:
      - name: l0_deficit_from_k
        direction: min
        threshold: 0.0
        role: target
        bar: goal
        reads_as: "k minus mean per-token L0 under the new !=0 counting; 0.0 = exactly k signed features survived (a ReLU'd variant would read ~32 at k=64)"
      - name: relu_topk_l0_predicate_delta
        direction: min
        threshold: 0.5
        role: guardrail
        bar: goal
        reads_as: "absolute per-token L0 shift on the ReLU-family control from swapping >0 for !=0; correct arithmetic gives exactly 0.0, and the 0.5 bound (half an active feature per token) is a real gate — any wholesale predicate regression (>= 0 counting padding zeros, < 0 counting nothing) shifts L0 by >= 1.0, up to the full k=64, and fails it"
      - name: bidirectionality_deviation
        direction: min
        threshold: 0.0084
        role: diagnostic
        reads_as: "|negative share of active features - 0.5|; 3-sigma band from 500 tokens x k=64 = 32000 signs (sigma = 0.002795)"
      - name: old_predicate_undercount_ratio
        direction: min
        threshold: 0.9
        role: diagnostic
        reads_as: "old >0-counted L0 over new !=0-counted L0 on AbsTopK acts; must be <= 0.9 to demonstrate a real undercount — expected ~0.5 (ratio = 1 - negative share); 1.0 would mean no negative actives at all (sign-killing), i.e. the undercount the fix corrects does not exist"
      - name: magnitude_selection_exact
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "share of tokens whose kept set equals exactly the k largest-magnitude pre-activations"
      - name: sign_preservation_exact
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "share of active features whose output sign matches their pre-activation sign"
      - name: abstopk_registered
        direction: max
        threshold: 1.0
        role: diagnostic
        bar: sanity
        reads_as: "1.0 when sae_lens.registry resolves abstopk to AbsTopKSAE and AbsTopKTrainingSAE"
    held_constant:
      - "model gpt2 via HookedTransformer on CPU, hook blocks.6.hook_resid_post, first 500 token positions of a fixed text"
      - "k=64, d_sae=8192, d_in=768 — the PR pre-registered sweep point (VALIDATION.md)"
      - "seed 0 for the random encoder projection; identical pre-activations fed to AbsTopK and the TopK/ReLU control"
      - "float32 CPU tensors, single deterministic pass, no wall-clock timing"
    avoid:
      - "no reconstruction-quality numbers: the encoder is random-init so mse/ce_loss_score/explained_variance would be meaningless and are not printed"
      - "the run_evals/ActivationsStore loop itself is not driven (ActivationsStore/ActivationScaler constructor signatures are outside the visible context); the changed !=0/>0 predicates are measured on the real AbsTopK operator output instead"
      - "no W&B logging — the training-parity run is deferred by the PR itself"
      - "no wall-clock timing on shared CPU; no unpinned live APIs (gpt2 weights fetched once from HF and cached)"
    report:
      headline: "Mechanism checks hold on real gpt2 activations: exactly-k magnitude selection with signs preserved, ~even sign split, and the predicate swap is an exact no-op for ReLU variants — parity vs TopK/JumpReLU remains deferred"
      findings:
        - "per-token L0 under the new !=0 counting falls {l0_deficit_from_k} short of k=64 (0.0 expected; ~32 would indicate sign-killing)"
        - "the >0 -> !=0 swap changes TopK/ReLU L0 by {relu_topk_l0_predicate_delta} (exactly 0.0 expected; guardrail bound 0.5, so any wholesale predicate regression reading >= 1.0 fails)"
        - "the negative share of active features sits {bidirectionality_deviation} from 0.5 (3-sigma band 0.0084)"
        - "the old >0 counting reports {old_predicate_undercount_ratio} of the new L0 on AbsTopK acts (~0.5 = the corrected 2x undercount; bound 0.9, 1.0 = no negative actives)"
        - "{magnitude_selection_exact} of tokens keep exactly the k largest-magnitude pre-activations"
        - "{sign_preservation_exact} of active features retain their pre-activation sign"
        - "{abstopk_registered} registry wiring for architecture abstopk"
      establishes:
        - "the ported operator implements the paper's selection rule — top-k by magnitude, sign preserved, exactly k active per token — on real residual-stream activations at the PR's pre-registered geometry"
        - "the eval-counting fix counts negative-active features and is an exact no-op for ReLU-family activations at the predicate level"
      does_not_establish:
        - "reconstruction/CE parity with TopK or JumpReLU at matched L0 — the PR defers that to a W&B run that has not happened"
        - "any interpretability claim: probing, steering, or single features encoding contrasting concepts"
        - "behaviour of the full get_sparsity_and_variance_metrics loop through ActivationsStore — only its changed predicates are exercised"
        - "the latent issue that evals.py l1 is a signed sum and cancels toward 0 for AbsTopK (untouched by this PR)"
      not_measured:
        - "mse, ce_loss_score, explained_variance, dead-feature counts, training dynamics, parameter-count parity with TopK, throughput/wall-clock"
      caveat: "the paper's headline results (7 probing/steering tasks across 4 LLMs, matching supervised DiM) have no harness in this repo; only the mechanism and eval-counting slices of the claim are testable now"
      next: "run the deferred W&B training-parity protocol: shared activation cache for gpt2/pythia-160m, k in {16,64,256}, TopK/JumpReLU controls, logging run_evals l0/mse/ce_loss_score plus dead-feature count"
    provenance:
      suite: "synthesized"
      held_constant: "protocol_doc:VALIDATION.md (PR diff pre-registers gpt2, blocks.*.hook_resid_post, d_sae=8192, k in {16,64,256})"
      l0_deficit_from_k: "inferred (claim arithmetic: exactly-k scatter of signed values; deficit > 0 only if a selected pre-activation is exactly 0.0)"
      relu_topk_l0_predicate_delta: "inferred (claim arithmetic: (t>0) == (t!=0) elementwise for t >= 0 gives exactly 0.0; bound 0.5 = half an active feature per token, chosen between exact correctness (0.0) and the smallest systematic regression (1 full feature/token, up to k=64 for a wrong operator/tensor) so the guardrail can actually fail)"
      old_predicate_undercount_ratio: "inferred (ratio = positive actives / all actives = 1 - negative share; expected ~0.5 with the same 3-sigma band as bidirectionality_deviation; bound 0.9 fails only when neg_share < 0.10, i.e. essentially no undercount for the fix to correct)"
      bidirectionality_deviation: "inferred (3-sigma band 0.008385 at 32000 signs on symmetric pre-activations)"
      baseline_fallback: "inferred (diff-added AbsTopK imported defensively; on the baseline arm the pathway degrades to the pre-change ReLU-family control: abstopk_registered 0.0, old_predicate_undercount_ratio 1.0, bidirectionality_deviation 0.5 — no crash, guardrail still passes at exactly 0.0 against its 0.5 bound, JSON line still prints)"
      compute: "inferred (mechanism/numerics claim, no device claim; GPU parity explicitly deferred by the PR)"
```